# Studi Kasus 3 — Prioritas Inspeksi Aset Berbasis GIS
**Materi:** Career and Competition in Industrial Era with GIS  
Kasus ini mencontohkan penggunaan koordinat, analisis spasial, dan data operasional untuk menyusun prioritas inspeksi aset.

## Rumusan masalah dan batasan
Tim fasilitas fiktif perlu menentukan urutan inspeksi aset. Skor menggabungkan kondisi, kritikalitas, dan jarak dari depot. Semua lokasi sintetis; jarak garis lurus bukan waktu tempuh jalan dan skor bukan standar keselamatan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(73)
depot = {"nama": "Depot Tim", "lat": -3.80, "lon": 102.26}
n = 28
assets = pd.DataFrame({
    "asset_id": [f"AST-{i:03d}" for i in range(1, n+1)],
    "lat": depot["lat"] + rng.uniform(-0.12, 0.12, n),
    "lon": depot["lon"] + rng.uniform(-0.14, 0.14, n),
    "kondisi": rng.choice(["baik", "sedang", "buruk"], n, p=[0.35, 0.4, 0.25]),
    "kritikalitas": rng.integers(1, 6, n)
})
assets.head()

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    
    r = 6371.0
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat, dlon = lat2-lat1, lon2-lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

assets["jarak_depot_km"] = assets.apply(
    lambda r: haversine_km(depot["lat"], depot["lon"], r["lat"], r["lon"]), axis=1
)
condition_points = {"baik": 1, "sedang": 3, "buruk": 5}
assets["skor_kondisi"] = assets["kondisi"].map(condition_points)
# Aturan skor edukatif: kondisi buruk dan kritikalitas tinggi menaikkan prioritas;
# jarak hanya menjadi faktor kecil, bukan pengganti penilaian keselamatan.
assets["skor_prioritas"] = (
    assets["skor_kondisi"] * 0.5
    + assets["kritikalitas"] * 0.4
    + (assets["jarak_depot_km"] / assets["jarak_depot_km"].max()) * 0.1 * 5
)
assets = assets.sort_values("skor_prioritas", ascending=False).reset_index(drop=True)
display(assets.head(10).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(assets["lon"], assets["lat"], c=assets["skor_prioritas"],
                s=40 + assets["kritikalitas"]*18, alpha=0.8)
ax.scatter(depot["lon"], depot["lat"], marker="*", s=250, label="Depot")
for _, row in assets.head(5).iterrows():
    ax.annotate(row["asset_id"], (row["lon"], row["lat"]), fontsize=8)
ax.set_title("Sebaran aset sintetis dan skor prioritas")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.colorbar(sc, ax=ax, label="Skor prioritas")
plt.show()

In [ ]:
print("Aset yang perlu ditinjau terlebih dahulu:")
display(assets[["asset_id", "kondisi", "kritikalitas", "jarak_depot_km", "skor_prioritas"]].head(8))
print("\nValidasi koordinat:")
print("Latitude valid:", assets["lat"].between(-90, 90).all())
print("Longitude valid:", assets["lon"].between(-180, 180).all())

## Interpretasi
Prioritas teratas adalah keluaran dari bobot yang ditentukan pada notebook, bukan kebenaran objektif. Untuk penerapan nyata, tambahkan riwayat kegagalan, konsekuensi risiko, akses jalan, waktu tempuh, dan verifikasi petugas lapangan. Lindungi koordinat aset sensitif sebelum membagikan repositori.